# Train frame — chapter 4

Frame-level classification on the canonical JSONL. Wav2Vec2 base + a per-frame head.
Same `EpochCheckpointCallback` + two-phase pattern as chapter 3 (TRAIN→DEV, TRAIN+DEV→TEST).

**Input**
- A canonical JSONL with sequence labels (e.g. `data/processed_jsonl/gos_frame.jsonl`).
  Each record carries `frame_rate_hz` and `labels[label_key]` is a list of ints aligned to that rate.

**Output**
- `runs/{dataset}_{label_key}_{task_type}_{timestamp}/`
  - `phase1_dev/` — TRAIN→DEV per-epoch logs (no model saved)
  - `phase2_test/` — TRAIN+DEV→TEST per-epoch logs
- `models/{dataset}_{label_key}_{task_type}_{timestamp}/best_model/` — phase-2 best epoch's weights.

**Hard constraints (v1)**
- `frame_rate_hz == 50` only. Wav2Vec2's CNN front-end strides exactly 320 samples at 16 kHz → 50 Hz model output. Any other rate is rejected loudly. Resampling labels is intentionally deferred (cleanliness > flexibility for v1).
- `task_type == "classification"` only. The Config has a `task_type` flag for parity with chapter 3, but `"regression"` raises `NotImplementedError`. Frame-level regression is a future chapter.
- `len(label_order) >= 2`. Binary stress is `[0, 1]`.

**What's shared with chapter 3**
- Canonical JSONL loader (`udp.iter_jsonl` filtered by `split`).
- Audio preprocessing path (`soundfile.read` → `feature_extractor` without padding; collator pads).
- `EpochCheckpointCallback` (logs per-epoch metrics + predictions + a plot; no weights).
- Two-phase TRAIN→DEV / TRAIN+DEV→TEST with phase-2 saving the best model.
- Config dataclass at the top, test mode mirrors output under `runs/test/` + `models/test/`.

**What's different from chapter 3**
- Model head outputs `(B, T, num_labels)` instead of `(B, num_labels)`. Custom `Wav2Vec2ForFrameClassification` modeled on parlastress.
- Loss is token-level cross-entropy with `ignore_index=-100`. Pad frames are excluded.
- Labels are list-valued; the collator pads them to the batch max with `-100`.
- Metrics: `frame_accuracy`, `frame_macro_f1`, `frame_f1_positive` (positive-class F1; the binary-stress headline). All computed only on non-pad frames.
- Per-epoch plot: N example gold-vs-pred strips, not a confusion matrix. Boundary / IoU / event-level analysis is deferred to chapter 5.

---

## 0. Setup

In [1]:
import os
import sys
from pathlib import Path

# Find PROJECT_ROOT via utils_dataprep (chapter 1 already does this).
HERE = Path.cwd()
if HERE.name != "4_frame_models":
    candidate = HERE / "4_frame_models"
    if candidate.exists():
        HERE = candidate
sys.path.insert(0, str(HERE.parent / "1_data_prep"))

import utils_dataprep as udp
PROJECT_ROOT = udp.PROJECT_ROOT
print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"Chapter dir = {HERE}")

PROJECT_ROOT = /home/ivan/Posao_IJS/Stepping_Stones/github_full_repo/slavic-speech-pipeline
Chapter dir = /home/ivan/Posao_IJS/Stepping_Stones/github_full_repo/slavic-speech-pipeline/4_frame_models


Standard third-party imports.

In [2]:
import json
import shutil
from collections import Counter
from dataclasses import dataclass, field
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import soundfile as sf
import torch
import torch.nn as nn
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoConfig, AutoFeatureExtractor,
    Trainer, TrainerCallback, TrainingArguments,
    Wav2Vec2Model, Wav2Vec2PreTrainedModel,
)

plt.rcParams["figure.dpi"] = 100

---

## 1. Config

All knobs are here.

**Important rules**
- `task_type`: `"classification"` only in v1. `"regression"` raises `NotImplementedError` — frame-level regression is a future chapter.
- `label_key`: which key inside `labels` holds the per-frame sequence (e.g. `"primary_stress"`).
- `label_order`: **required.** Canonical ordering of the label space. Binary stress = `[0, 1]`. Used as `id2label` *and* for stable F1 reporting. No alphabetical fallback.
- `model_name`: defaults to `facebook/wav2vec2-base` (~95 M, parlastress-style). Test mode swaps in the tiny random model.
- `use_cuda`: **False by default.** Flip True on GPU.
- `test_mode`: tiny model, 1 epoch, batch 2, output under `runs/test/` + `models/test/`.

**Audio length cap.** `enable_max_audio_seconds` is OFF by default. When ON, records longer than `max_audio_seconds` are **dropped** with a warning. They are NOT truncated — truncating audio without also truncating frame labels would misalign them, and truncating both halfway through a stressed region would silently corrupt training signal. Drop is the safe default.

**Output layout** (matches chapter 3)
- `runs/<run_name>/` — per-epoch logs, predictions, plots, `config.json`. Lightweight.
- `models/<run_name>/best_model/` — phase-2 best model weights + artifacts. Heavy, gitignored.

In [3]:
@dataclass
class Config:
    # -- Data ----------------------------------------------------------------
    jsonl_path: str = "data/processed_jsonl/rog_frame.raw.jsonl"
    label_key: str  = "filled_pause"
    task_type: str  = "classification"   # only "classification" in v1; "regression" raises

    # REQUIRED canonical label order. Binary stress = [0, 1].
    # No alphabetical fallback — ints/strings, but specified.
    label_order: list = field(default_factory=lambda: [0, 1])

    # -- Frame alignment -----------------------------------------------------
    # Wav2Vec2 strides 320 samples at 16 kHz → 50 Hz model output. Any other
    # rate is a hard error; resampling labels is deferred to a future version.
    required_frame_rate_hz: int = 50

    # -- Model ---------------------------------------------------------------
    model_name: str             = "facebook/wav2vec2-base"
    freeze_feature_encoder: bool = True
    head_dropout: float         = 0.1

    # -- Training ------------------------------------------------------------
    batch_size: int      = 8
    grad_accum: int      = 2
    learning_rate: float = 1e-5
    num_epochs: int      = 20
    max_grad_norm: float = 1.0
    warmup_ratio: float  = 0.10

    # -- Output --------------------------------------------------------------
    runs_dir: str   = "runs"
    models_dir: str = "models"

    # -- Best-epoch selection ------------------------------------------------
    # "frame_macro_f1" | "frame_accuracy" | "frame_f1_positive"
    best_metric: str = "frame_macro_f1"

    # -- Audio length cap ----------------------------------------------------
    # OFF by default; when ON, longer records are DROPPED (never truncated).
    enable_max_audio_seconds: bool = False
    max_audio_seconds: float       = 10.0

    # -- Preprocessing -------------------------------------------------------
    # batch_size for the .map() that loads + feature-extracts. 1 keeps RAM
    # predictable; bump on machines with RAM to spare.
    preprocess_batch_size: int  = 1
    dataloader_num_workers: int = 0

    # -- Hardware ------------------------------------------------------------
    use_cuda: bool   = False
    cuda_device: str = "0"

    # -- Visualization -------------------------------------------------------
    # How many records from the first N of the eval set to plot as gold/pred
    # strips on a single PNG per epoch.
    n_examples_to_plot: int = 6

    # -- Test mode -----------------------------------------------------------
    test_mode: bool       = True                                                                ######### TEST MODE
    test_n_train: int     = 16
    test_n_dev: int       = 4
    test_n_test: int      = 4
    test_model_name: str  = "hf-internal-testing/tiny-random-wav2vec2"
    test_num_epochs: int  = 1
    test_batch_size: int  = 2


cfg = Config()

# Apply test-mode clamps
if cfg.test_mode:
    udp.banner("🧪 TEST MODE", char="-")
    cfg.model_name = cfg.test_model_name
    cfg.num_epochs = cfg.test_num_epochs
    cfg.batch_size = cfg.test_batch_size
    cfg.grad_accum = 1
    cfg.runs_dir   = "runs/test"
    cfg.models_dir = "models/test"

# Device resolution
if cfg.use_cuda and torch.cuda.is_available():
    os.environ["CUDA_VISIBLE_DEVICES"] = cfg.cuda_device
    DEVICE = "cuda"
elif cfg.use_cuda and not torch.cuda.is_available():
    print("⚠️  use_cuda=True but no CUDA device available; falling back to CPU")
    DEVICE = "cpu"
else:
    DEVICE = "cpu"

print(cfg)
print(f"device = {DEVICE}")


----------------------------------------------------------------------
🧪 TEST MODE
----------------------------------------------------------------------
Config(jsonl_path='data/processed_jsonl/rog_frame.raw.jsonl', label_key='filled_pause', task_type='classification', label_order=[0, 1], required_frame_rate_hz=50, model_name='hf-internal-testing/tiny-random-wav2vec2', freeze_feature_encoder=True, head_dropout=0.1, batch_size=2, grad_accum=1, learning_rate=1e-05, num_epochs=1, max_grad_norm=1.0, warmup_ratio=0.1, runs_dir='runs/test', models_dir='models/test', best_metric='frame_macro_f1', enable_max_audio_seconds=False, max_audio_seconds=10.0, preprocess_batch_size=1, dataloader_num_workers=0, use_cuda=False, cuda_device='0', n_examples_to_plot=6, test_mode=True, test_n_train=16, test_n_dev=4, test_n_test=4, test_model_name='hf-internal-testing/tiny-random-wav2vec2', test_num_epochs=1, test_batch_size=2)
device = cpu


---

## 2. Validate the config before doing real work

Fail loud here so you don't burn 4 hours of training only to learn `label_order` was missing or `task_type="regression"`.

In [4]:
def validate_config(cfg: Config) -> None:
    # task_type
    if cfg.task_type == "regression":
        raise NotImplementedError(
            "Frame-level regression is deferred to a future chapter. "
            "v1 supports task_type='classification' only — see FUTURE.md."
        )
    if cfg.task_type != "classification":
        raise ValueError(f"task_type must be 'classification', got {cfg.task_type!r}")

    # label_order
    if not cfg.label_order or len(cfg.label_order) < 2:
        raise ValueError(
            "Config.label_order is REQUIRED and must have at least 2 entries. "
            "Binary stress = [0, 1]. No alphabetical fallback."
        )
    if len(set(cfg.label_order)) != len(cfg.label_order):
        raise ValueError(f"label_order has duplicates: {cfg.label_order}")

    # frame rate
    if cfg.required_frame_rate_hz != 50:
        raise ValueError(
            f"required_frame_rate_hz must be 50 in v1 (Wav2Vec2 stride = 320 samples @ 16 kHz). "
            f"Got {cfg.required_frame_rate_hz}. Resampling labels is deferred."
        )

    # best_metric
    if cfg.best_metric not in ("frame_macro_f1", "frame_accuracy", "frame_f1_positive"):
        raise ValueError(f"best_metric: invalid {cfg.best_metric!r}")


validate_config(cfg)
print("✅ config valid")

✅ config valid


---

## 3. Load JSONL, filter to records that carry `label_key`

Records missing the target label are silently dropped — chapter-2 sniff would have already alerted you if this is a big fraction.

We also drop records whose `labels[label_key]` isn't a non-empty list (i.e., scalar-labeled records that wandered into a frame JSONL). Frame mode wants sequences.

In [5]:
def load_split(jsonl_path: str, split: str, label_key: str) -> list[dict]:
    out = []
    for r in udp.iter_jsonl(jsonl_path):
        if r["split"] != split:
            continue
        v = r.get("labels", {}).get(label_key)
        if v is None:
            continue
        if not isinstance(v, list):
            # Scalar label in a frame run — skip and warn once per file_id at most.
            continue
        if len(v) == 0:
            continue
        out.append(r)
    return out


train_records = load_split(cfg.jsonl_path, "train", cfg.label_key)
dev_records   = load_split(cfg.jsonl_path, "dev",   cfg.label_key)
test_records  = load_split(cfg.jsonl_path, "test",  cfg.label_key)

if cfg.test_mode:
    train_records = train_records[: cfg.test_n_train]
    dev_records   = dev_records[:   cfg.test_n_dev]
    test_records  = test_records[:  cfg.test_n_test]

print(f"train: {len(train_records)}")
print(f"dev:   {len(dev_records)}")
print(f"test:  {len(test_records)}")
if not train_records or not dev_records or not test_records:
    raise ValueError("one of the splits is empty after filtering for label_key — check the JSONL")

train: 16
dev:   4
test:  4


---

## 4. Validate `frame_rate_hz == 50` on every record

This is the cleanliness gate. v1 supports only 50 Hz labels because Wav2Vec2's native frame rate is exactly that — every other rate would require interpolation that's easy to get subtly wrong and hard to debug. Hard fail here, with a precise count of offenders.

In [6]:
def validate_frame_rate(records: list[dict], required_hz: int) -> None:
    bad = []
    missing = []
    for r in records:
        fr = r.get("frame_rate_hz")
        if fr is None:
            missing.append(r["instance_id"])
        elif fr != required_hz:
            bad.append((r["instance_id"], fr))

    if missing:
        head = missing[:5]
        raise ValueError(
            f"{len(missing)} records are missing frame_rate_hz. Examples: {head}. "
            f"Every frame-level record must declare its rate."
        )
    if bad:
        head = bad[:5]
        rates_seen = sorted({b[1] for b in bad})
        raise ValueError(
            f"{len(bad)} records have frame_rate_hz != {required_hz}. "
            f"Rates seen: {rates_seen}. Examples: {head}. "
            f"v1 supports only {required_hz} Hz; resampling labels is deferred."
        )


for split_name, recs in [("train", train_records), ("dev", dev_records), ("test", test_records)]:
    validate_frame_rate(recs, cfg.required_frame_rate_hz)
print(f"✅ all records carry frame_rate_hz={cfg.required_frame_rate_hz}")

✅ all records carry frame_rate_hz=50


---

## 5. Audio length cap (optional)

Only runs if `enable_max_audio_seconds=True`. Drops records whose audio is longer than the cap.

**Why drop, not truncate?** Truncating audio without identically truncating labels misaligns them (off-by-N frames silently corrupts training). Truncating both correctly is straightforward, but truncating *mid-stress* changes the label distribution in a way that's hard to reason about. For v1 we drop and warn.

In [7]:
def drop_long_records(records: list[dict], max_s: float) -> tuple[list[dict], int]:
    kept = []
    dropped = 0
    for r in records:
        try:
            dur = udp.get_wav_duration(r["audio_path"])
        except Exception as e:
            print(f"⚠️  could not read duration for {r['instance_id']}: {e}; keeping")
            kept.append(r)
            continue
        if dur > max_s:
            dropped += 1
            continue
        kept.append(r)
    return kept, dropped


if cfg.enable_max_audio_seconds:
    udp.banner(f"applying max_audio_seconds = {cfg.max_audio_seconds}s", char="-")
    train_records, n_tr_drop = drop_long_records(train_records, cfg.max_audio_seconds)
    dev_records,   n_dv_drop = drop_long_records(dev_records,   cfg.max_audio_seconds)
    test_records,  n_te_drop = drop_long_records(test_records,  cfg.max_audio_seconds)
    print(f"dropped (train={n_tr_drop}, dev={n_dv_drop}, test={n_te_drop})")
    print(f"kept    (train={len(train_records)}, dev={len(dev_records)}, test={len(test_records)})")
    if not train_records or not dev_records or not test_records:
        raise ValueError("a split became empty after applying max_audio_seconds")
else:
    print("max_audio_seconds: disabled (cfg.enable_max_audio_seconds=False)")

max_audio_seconds: disabled (cfg.enable_max_audio_seconds=False)


---

## 6. Build label mappings (stringified for HF) + per-split frame-label distribution

`label_order` is the source of truth. `label2id[label] = index in label_order`. Any class encountered in the frame sequences that isn't in `label_order` is a hard error.

HF transformers 5.x requires `label2id` *keys* to be strings in `from_pretrained` calls. We keep the internal mapping with native (int) keys and stringify only at the model-build boundary.

In [8]:
label2id = {lab: i for i, lab in enumerate(cfg.label_order)}
id2label = {i: lab for i, lab in enumerate(cfg.label_order)}
num_labels = len(cfg.label_order)

# Validate all frame labels are in label_order
seen = set()
for r in train_records + dev_records + test_records:
    seen.update(r["labels"][cfg.label_key])
unknown = seen - set(label2id)
if unknown:
    raise ValueError(
        f"Found frame labels not in Config.label_order: {sorted(unknown)}. "
        f"Either add them to label_order or fix the data."
    )

# Per-split frame-token counts (not per-record counts — frames)
def frame_counts(records: list[dict], label_key: str) -> Counter:
    c = Counter()
    for r in records:
        c.update(r["labels"][label_key])
    return c

print(f"Frame labels ({num_labels}, canonical order):")
print(f"   {'class':>10}  {'train':>10}  {'dev':>10}  {'test':>10}")
ftr = frame_counts(train_records, cfg.label_key)
fdv = frame_counts(dev_records,   cfg.label_key)
fte = frame_counts(test_records,  cfg.label_key)
for lab in cfg.label_order:
    print(f"   {str(lab):>10}  {ftr.get(lab, 0):>10d}  {fdv.get(lab, 0):>10d}  {fte.get(lab, 0):>10d}")
total_tr = sum(ftr.values()); total_dv = sum(fdv.values()); total_te = sum(fte.values())
print(f"   {'TOTAL':>10}  {total_tr:>10d}  {total_dv:>10d}  {total_te:>10d}")

Frame labels (2, canonical order):
        class       train         dev        test
            0     1494321      383232      403581
            1       17064        4003        2783
        TOTAL     1511385      387235      406364


---

## 7. Audio loading, feature extraction, and label alignment

`prepare_dataset_dict` turns canonical records into the list-of-dicts that `Dataset.from_list` wants: `audio_path`, `labels` (as a list), `instance_id`, etc.

`preprocess_function` (run via `.map(batched=True)`) does three things per record:

1. Loads the WAV with `soundfile` (16 kHz mono PCM-16 guaranteed by chapter 1; defensive resample if not).
2. Runs the feature extractor without padding (collator handles padding).
3. **Aligns the label sequence to the model's *actual* output frame count.** Labels are nominally at 50 Hz; Wav2Vec2-base outputs ~49 Hz (kernel-vs-stride boundary loss in the first CNN layer slightly drops the per-frame ratio below the nominal 320 samples/frame). To make label[t] correspond to model_frame[t] *exactly*, we resample the label sequence by nearest-neighbor index mapping to the model's true output length. This length is computed from the `Wav2Vec2Config`'s `conv_kernel` + `conv_stride` lists — the same arithmetic `Wav2Vec2Model._get_feat_extract_output_lengths` does internally. No instantiating a model just to call that helper.

**Why this matters.** Hard-coding `// 320` works for `facebook/wav2vec2-base` audio that's an exact multiple of 320 samples, but breaks subtly otherwise (off-by-one boundary loss per clip), and breaks badly for variants with different strides (e.g. `hf-internal-testing/tiny-random-wav2vec2`, which has stride 64). Aligning to the model's real output length is robust to all of these.

In [9]:
IGNORE_INDEX = -100


def compute_feat_extract_output_length(input_length: int,
                                       conv_kernel: list, conv_stride: list) -> int:
    """Replicate Wav2Vec2Model._get_feat_extract_output_lengths from config alone.

    For each CNN layer: out = (in - kernel) // stride + 1. Same arithmetic
    HuggingFace uses internally. Returns the number of output frames the model
    will produce for an input of `input_length` raw samples.
    """
    out = int(input_length)
    for k, s in zip(conv_kernel, conv_stride):
        out = (out - int(k)) // int(s) + 1
    return max(out, 0)


def align_labels_to_frames(label_seq, n_frames: int) -> list:
    """Resize a label sequence to length `n_frames` by nearest-neighbor index.

    Both the source labels (nominally at 50 Hz) and the model frames span the
    same audio duration; we map model-frame `j` to source-label index
    `(j * L) // n_frames`. Handles expansion and contraction symmetrically.
    """
    L = len(label_seq)
    if L == 0 or n_frames == 0:
        return []
    if L == n_frames:
        return list(label_seq)
    out = []
    for j in range(n_frames):
        i = (j * L) // n_frames
        if i >= L:
            i = L - 1
        out.append(label_seq[i])
    return out


def prepare_dataset_dict(records: list[dict], cfg: Config, label2id: dict) -> list[dict]:
    items = []
    for r in records:
        # Convert frame labels through label2id (no-op when they're already 0/1 ints
        # and label_order is [0, 1], but explicit for non-binary cases).
        raw = r["labels"][cfg.label_key]
        mapped = [label2id[v] for v in raw]
        items.append({
            "instance_id": r["instance_id"],
            "file_id":     r.get("file_id", ""),
            "audio_path":  str(udp.from_project_relative(r["audio_path"])),
            "labels":      mapped,
        })
    return items


def make_preprocess_function(feature_extractor, conv_kernel: list, conv_stride: list):
    """Factory: bind the model's CNN geometry into preprocess_function."""

    def preprocess_function(examples):
        """Load each WAV, run feature extractor, align labels to model frames."""
        audio_arrays = []
        for path in examples["audio_path"]:
            data, sr = sf.read(path, dtype="float32", always_2d=False)
            if data.ndim == 2:
                data = data.mean(axis=1)
            if sr != 16000:
                # Shouldn't happen if chapter 1 ran cleanly; resample as a fallback.
                import librosa
                data = librosa.resample(data, orig_sr=sr, target_sr=16000)
            audio_arrays.append(data)

        inputs = feature_extractor(
            audio_arrays, sampling_rate=16000, return_tensors=None, padding=False,
        )

        aligned_labels = []
        for input_values, label_seq in zip(inputs["input_values"], examples["labels"]):
            n_frames = compute_feat_extract_output_length(
                len(input_values), conv_kernel, conv_stride,
            )
            aligned_labels.append(align_labels_to_frames(label_seq, n_frames))

        return {"input_values": inputs["input_values"], "labels": aligned_labels}

    return preprocess_function

---

## 8. Data collator (pad audio + labels within each batch)

The feature extractor pads audio. We hand-pad labels with `-100` so token-CE skips them. Attention mask comes from the feature extractor.

In [10]:
class DataCollatorForFrame:
    def __init__(self, feature_extractor):
        self.feature_extractor = feature_extractor

    def __call__(self, features):
        input_values = [f["input_values"] for f in features]
        labels = [f["labels"] for f in features]

        batch = self.feature_extractor.pad(
            {"input_values": input_values},
            padding=True, return_tensors="pt", return_attention_mask=True,
        )

        max_label_len = max(len(l) for l in labels)
        padded_labels = []
        for l in labels:
            # Each l is a Python list (or 1-D tensor) — normalize to list
            if torch.is_tensor(l):
                l = l.tolist()
            else:
                l = list(l)
            padded_labels.append(l + [IGNORE_INDEX] * (max_label_len - len(l)))
        batch["labels"] = torch.tensor(padded_labels, dtype=torch.long)
        return batch

---

## 9. `Wav2Vec2ForFrameClassification`

Wav2Vec2 backbone + dropout + `Linear(hidden, num_labels)` applied per frame. Output shape `(B, T, num_labels)`. Loss is `nn.CrossEntropyLoss(ignore_index=-100)` over the flattened `(B*T, num_labels)` / `(B*T,)` views — pad frames are skipped automatically. Patterned after parlastress's frame head.

In [11]:
class Wav2Vec2ForFrameClassification(Wav2Vec2PreTrainedModel):
    """Wav2Vec2 + dropout + per-frame Linear classifier. Token-CE loss."""

    def __init__(self, config, head_dropout: float = 0.1):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.wav2vec2 = Wav2Vec2Model(config)
        self.dropout = nn.Dropout(head_dropout)
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)
        self.post_init()

    def forward(self, input_values, attention_mask=None, labels=None):
        outputs = self.wav2vec2(input_values, attention_mask=attention_mask)
        hidden = outputs.last_hidden_state           # (B, T, H)
        hidden = self.dropout(hidden)
        logits = self.classifier(hidden)             # (B, T, num_labels)

        loss = None
        if labels is not None:
            # labels: (B, T_labels). Align T to model output T.
            T_model = logits.shape[1]
            T_lab   = labels.shape[1]
            if T_lab > T_model:
                labels = labels[:, :T_model]
            elif T_lab < T_model:
                pad = torch.full(
                    (labels.shape[0], T_model - T_lab),
                    IGNORE_INDEX, dtype=labels.dtype, device=labels.device,
                )
                labels = torch.cat([labels, pad], dim=1)
            fct = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)
            loss = fct(
                logits.reshape(-1, self.num_labels),
                labels.reshape(-1),
            )
        return {"loss": loss, "logits": logits}


def build_model(cfg: Config, num_labels: int, label2id, id2label):
    # HF transformers 5.x requires label2id keys to be str on from_pretrained.
    hf_label2id = {str(k): int(v) for k, v in label2id.items()}
    hf_id2label = {int(k): str(v) for k, v in id2label.items()}

    config_obj = AutoConfig.from_pretrained(
        cfg.model_name,
        num_labels=num_labels,
        label2id=hf_label2id,
        id2label=hf_id2label,
    )
    model = Wav2Vec2ForFrameClassification(config_obj, head_dropout=cfg.head_dropout)
    model.wav2vec2 = Wav2Vec2Model.from_pretrained(
        cfg.model_name, config=config_obj, ignore_mismatched_sizes=True,
    )

    if cfg.freeze_feature_encoder:
        model.wav2vec2.freeze_feature_encoder()
        print("🔒 feature encoder (CNN) frozen")
    return model

---

## 10. Frame-level metrics

Computed on non-pad frames only — `-100` is the pad sentinel and is masked out before scoring.

- `frame_accuracy` — proportion of frames classified correctly.
- `frame_macro_f1` — unweighted mean of per-class F1. Headline for non-binary cases.
- `frame_f1_positive` — F1 of the **last** class in `label_order`. For binary stress (`[0, 1]`) this is the F1 of stressed frames, which is the headline. For `>2` classes it's whatever you put last — usually not what you want; set `best_metric="frame_macro_f1"` in that case.

> **v1 keeps frame-level metrics only.** Boundary metrics (region IoU, start/end timing tolerance) and event-level metrics (precision/recall of detected stress regions) live in chapter 5, computed from saved predictions. Adding them to the training loop would double the metrics surface and isn't where epoch-selection signal lives.

In [12]:
def compute_frame_metrics(eval_pred):
    """
    eval_pred.predictions: (B, T, num_labels) — logits
    eval_pred.label_ids  : (B, T) — gold labels with -100 padding
    """
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)        # (B, T)
    mask = labels != IGNORE_INDEX             # (B, T)
    y_true = labels[mask]
    y_pred = preds[mask]

    acc = float(accuracy_score(y_true, y_pred))
    macro_f1 = float(f1_score(y_true, y_pred, average="macro", zero_division=0))

    # Positive-class F1 = F1 of the last class id in label_order
    # (for binary [0, 1] this is class 1).
    # We compute against the full label-id range so absent classes report 0.0.
    n_classes = int(logits.shape[-1])
    per_class_f1 = f1_score(
        y_true, y_pred,
        labels=list(range(n_classes)),
        average=None, zero_division=0,
    )
    positive_id = n_classes - 1
    f1_pos = float(per_class_f1[positive_id])

    return {
        "frame_accuracy": acc,
        "frame_macro_f1": macro_f1,
        "frame_f1_positive": f1_pos,
    }

---

## 11. Per-epoch artifacts

For each epoch we save:

- `epoch_summary.json` — losses + frame metrics.
- `predictions.json` — per-record `instance_id`, gold frame sequence (trimmed to non-pad length), predicted frame sequence (trimmed to the same length).
- `example_predictions.png` — first `n_examples_to_plot` records as horizontal colored strips: gold on top, prediction below. Inspect at a glance whether the model is producing reasonable structure or just one-class blobs.

In [13]:
def save_predictions_json(predictions, labels, items, out_path: Path, id2label: dict):
    """
    predictions: (B, T, num_labels) or (B, T) — logits if 3-D, argmax if 2-D
    labels     : (B, T) with -100 pad
    items      : list of dicts (one per record in batch order)
    """
    if predictions.ndim == 3:
        pred_idx = np.argmax(predictions, axis=-1)
    else:
        pred_idx = predictions

    out = []
    for i, item in enumerate(items):
        gold_row = labels[i]
        pred_row = pred_idx[i]
        # Trim to non-pad length
        valid = gold_row != IGNORE_INDEX
        gold_seq = gold_row[valid].tolist()
        pred_seq = pred_row[valid].tolist()
        out.append({
            "instance_id": item["instance_id"],
            "file_id":     item.get("file_id", ""),
            "n_frames":    int(len(gold_seq)),
            "gold":        [id2label[int(g)] for g in gold_seq],
            "pred":        [id2label[int(p)] for p in pred_seq],
            "gold_raw":    [int(g) for g in gold_seq],
            "pred_raw":    [int(p) for p in pred_seq],
        })
    out_path.write_text(json.dumps(out, indent=2, ensure_ascii=False))

---

## 12. `plot_example_predictions` — gold vs pred strips

A single PNG with N pairs of horizontal strips: gold above, pred below, for the first N records of the eval set. Colors come from a small qualitative palette (extend it if you have more than 8 classes). Pad frames are not drawn.

In [14]:
# A simple categorical palette. Extend if num_labels > 8.
_PALETTE = ["#dddddd", "#d62728", "#1f77b4", "#2ca02c", "#9467bd",
            "#ff7f0e", "#8c564b", "#e377c2"]


def _row_to_image(seq, n_classes: int):
    """Turn a 1-D label sequence into an (1, T) RGBA image via palette."""
    pal = (_PALETTE * ((n_classes // len(_PALETTE)) + 1))[:n_classes]
    rgba = np.zeros((1, len(seq), 4), dtype=float)
    for t, v in enumerate(seq):
        rgba[0, t] = mcolors.to_rgba(pal[int(v)])
    return rgba, pal


def plot_example_predictions(predictions, labels, items, out_path: Path,
                             id2label: dict, n_examples: int):
    if predictions.ndim == 3:
        pred_idx = np.argmax(predictions, axis=-1)
    else:
        pred_idx = predictions

    n = min(n_examples, len(items))
    if n == 0:
        return

    fig, axes = plt.subplots(n, 1, figsize=(10, max(2, 0.9 * n)), squeeze=False)
    axes = axes[:, 0]
    n_classes = len(id2label)

    for i in range(n):
        gold_row = labels[i]
        pred_row = pred_idx[i]
        valid = gold_row != IGNORE_INDEX
        gold_seq = gold_row[valid]
        pred_seq = pred_row[valid]

        gold_img, pal = _row_to_image(gold_seq, n_classes)
        pred_img, _   = _row_to_image(pred_seq, n_classes)
        # Stack gold (top) and pred (bottom) into a (2, T, 4) image.
        strip = np.concatenate([gold_img, pred_img], axis=0)
        ax = axes[i]
        ax.imshow(strip, aspect="auto", interpolation="nearest")
        ax.set_yticks([0, 1])
        ax.set_yticklabels(["gold", "pred"], fontsize=8)
        ax.set_xticks([])
        title = items[i]["instance_id"]
        if len(title) > 60:
            title = "..." + title[-57:]
        ax.set_title(title, fontsize=8, loc="left")

    # Legend (single, at the bottom)
    handles = [
        plt.Rectangle((0, 0), 1, 1, color=pal[k]) for k in range(n_classes)
    ]
    labels_str = [str(id2label[k]) for k in range(n_classes)]
    fig.legend(handles, labels_str, loc="lower center", ncol=min(n_classes, 6),
               fontsize=8, bbox_to_anchor=(0.5, -0.02))
    plt.tight_layout(rect=[0, 0.03, 1, 1])
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)

---

## 13. `EpochCheckpointCallback`

Evaluates the eval set every epoch, writes per-epoch logs. Does **not** save model weights — only the best epoch's model is saved, and only in phase 2.

In [15]:
class EpochCheckpointCallback(TrainerCallback):
    def __init__(self, phase_dir: Path, eval_dataset, eval_items,
                 compute_metrics, data_collator, cfg: Config,
                 label_order: list, id2label: dict):
        self.phase_dir = phase_dir
        self.logs_dir = phase_dir / "epoch_logs"
        self.logs_dir.mkdir(parents=True, exist_ok=True)
        self.eval_dataset = eval_dataset
        self.eval_items = eval_items
        self.compute_metrics = compute_metrics
        self.data_collator = data_collator
        self.cfg = cfg
        self.label_order = label_order
        self.id2label = id2label
        self.epoch_results: list[dict] = []

    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        epoch = int(state.epoch)
        print(f"\n💾 logging epoch {epoch}…")
        epoch_dir = self.logs_dir / f"epoch_{epoch}"
        epoch_dir.mkdir(parents=True, exist_ok=True)

        eval_trainer = Trainer(
            model=model, args=args,
            compute_metrics=self.compute_metrics,
            data_collator=self.data_collator,
        )
        out = eval_trainer.predict(self.eval_dataset)

        # HF predict uses 'test_' prefix; normalize to 'eval_'.
        metrics = {k.replace("test_", "eval_"): v for k, v in out.metrics.items()}

        # Latest train loss
        train_loss = None
        for log in reversed(state.log_history):
            if "loss" in log:
                train_loss = log["loss"]; break

        epoch_info = {"epoch": epoch, "train_loss": train_loss, **metrics}
        self.epoch_results.append(epoch_info)

        # predictions.json
        save_predictions_json(
            out.predictions, out.label_ids, self.eval_items,
            epoch_dir / "predictions.json",
            id2label=self.id2label,
        )
        # example strips
        plot_example_predictions(
            out.predictions, out.label_ids, self.eval_items,
            epoch_dir / "example_predictions.png",
            id2label=self.id2label,
            n_examples=self.cfg.n_examples_to_plot,
        )

        (epoch_dir / "epoch_summary.json").write_text(json.dumps(epoch_info, indent=2))

        print(
            f"   epoch={epoch}  "
            f"macroF1={metrics.get('eval_frame_macro_f1', 0):.4f}  "
            f"acc={metrics.get('eval_frame_accuracy', 0):.4f}  "
            f"F1+={metrics.get('eval_frame_f1_positive', 0):.4f}"
        )

---

## 14. `run_phase` — train + evaluate + save logs

One function used by both phases. The only differences between phase 1 and phase 2 are:
- Which records make up the train set.
- Which split is the eval set.
- Whether the best model is saved.

In [16]:
def best_epoch_of(epoch_results: list[dict], cfg: Config) -> dict:
    # All three current metrics are higher-is-better.
    m = cfg.best_metric
    return max(
        epoch_results,
        key=lambda r: (r.get(f"eval_{m}", float("-inf"))
                       if r.get(f"eval_{m}") is not None else float("-inf")),
    )


def run_phase(*, phase_name: str, train_records: list[dict], eval_records: list[dict],
              eval_split_name: str, save_best_model: bool,
              cfg: Config, run_dir: Path, model_dir: Path, feature_extractor,
              label2id, id2label) -> tuple[list[dict], dict]:
    udp.banner(f"PHASE: {phase_name}  (train→{eval_split_name})")
    phase_dir = run_dir / phase_name
    phase_dir.mkdir(parents=True, exist_ok=True)

    # Load the model's config first so we can compute output-frame counts
    # before .map(). conv_kernel + conv_stride define the CNN geometry; these
    # are static config fields, no weight download involved beyond the config.
    print(f"loading config: {cfg.model_name}")
    model_config = AutoConfig.from_pretrained(cfg.model_name)
    conv_kernel = list(model_config.conv_kernel)
    conv_stride = list(model_config.conv_stride)
    print(f"   conv_kernel = {conv_kernel}")
    print(f"   conv_stride = {conv_stride}")

    # Build items + datasets
    train_items = prepare_dataset_dict(train_records, cfg, label2id)
    eval_items  = prepare_dataset_dict(eval_records,  cfg, label2id)
    train_ds = Dataset.from_list(train_items)
    eval_ds  = Dataset.from_list(eval_items)

    # Preprocess
    preprocess_fn = make_preprocess_function(feature_extractor, conv_kernel, conv_stride)
    print(f"preprocessing {len(train_ds)} train + {len(eval_ds)} eval (batch_size={cfg.preprocess_batch_size})…")
    train_ds = train_ds.map(
        preprocess_fn,
        batched=True, batch_size=cfg.preprocess_batch_size,
        remove_columns=train_ds.column_names,
    )
    eval_ds = eval_ds.map(
        preprocess_fn,
        batched=True, batch_size=cfg.preprocess_batch_size,
        remove_columns=eval_ds.column_names,
    )
    train_ds.set_format(type="torch", columns=["input_values", "labels"])
    eval_ds.set_format(type="torch",  columns=["input_values", "labels"])

    # Model
    print(f"building model: {cfg.model_name}")
    model = build_model(cfg, num_labels=len(cfg.label_order),
                        label2id=label2id, id2label=id2label)

    # Collator + metrics
    data_collator = DataCollatorForFrame(feature_extractor)
    compute_metrics = compute_frame_metrics

    # Warmup
    steps_per_epoch = max(1, len(train_ds) // (cfg.batch_size * cfg.grad_accum))
    total_steps = steps_per_epoch * cfg.num_epochs
    warmup_steps = int(total_steps * cfg.warmup_ratio)

    training_args = TrainingArguments(
        output_dir=str(phase_dir / "trainer_tmp"),
        eval_strategy="no",
        save_strategy="no",
        logging_strategy="steps",
        logging_steps=10,
        report_to="none",
        label_names=["labels"],
        per_device_train_batch_size=cfg.batch_size,
        per_device_eval_batch_size=cfg.batch_size,
        num_train_epochs=cfg.num_epochs,
        gradient_accumulation_steps=cfg.grad_accum,
        learning_rate=cfg.learning_rate,
        warmup_steps=warmup_steps,
        lr_scheduler_type="linear",
        max_grad_norm=cfg.max_grad_norm,
        remove_unused_columns=False,
        use_cpu=(DEVICE == "cpu"),
        dataloader_num_workers=cfg.dataloader_num_workers,
    )

    callback = EpochCheckpointCallback(
        phase_dir=phase_dir, eval_dataset=eval_ds, eval_items=eval_items,
        compute_metrics=compute_metrics, data_collator=data_collator, cfg=cfg,
        label_order=cfg.label_order, id2label=id2label,
    )

    trainer = Trainer(
        model=model, args=training_args,
        train_dataset=train_ds, eval_dataset=eval_ds,
        compute_metrics=compute_metrics,
        data_collator=data_collator, callbacks=[callback],
    )
    print(f"🚀 training {cfg.num_epochs} epochs (bs={cfg.batch_size} ga={cfg.grad_accum} lr={cfg.learning_rate})")
    trainer.train()

    # Phase summary
    (phase_dir / "all_epochs_summary.json").write_text(
        json.dumps(callback.epoch_results, indent=2)
    )
    best = best_epoch_of(callback.epoch_results, cfg)
    print(f"\n🏆 best epoch in {phase_name}: {best['epoch']}")
    for k, v in best.items():
        if isinstance(v, float):
            print(f"   {k}: {v:.4f}")
        else:
            print(f"   {k}: {v}")

    # Save best model (phase 2 only)
    if save_best_model:
        best_dir = model_dir / "best_model"
        best_dir.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(best_dir)
        feature_extractor.save_pretrained(best_dir)
        src = phase_dir / "epoch_logs" / f"epoch_{best['epoch']}"
        if src.exists():
            for f in src.iterdir():
                shutil.copy(f, best_dir / f.name)
        info_lines = [
            f"run_name: {run_dir.name}",
            f"run_dir:  {run_dir}",
            f"phase:    {phase_name}",
            f"epoch:    {best['epoch']}",
        ]
        (best_dir / "run_info.txt").write_text("\n".join(info_lines) + "\n")
        print(f"   saved best model → {best_dir.relative_to(PROJECT_ROOT)}")

    shutil.rmtree(phase_dir / "trainer_tmp", ignore_errors=True)
    return callback.epoch_results, best

---

## 15. Set up the run directory

`runs/{dataset}_{label_key}_{task_type}_{timestamp}/`. The dataset name is read from the first record.

In [17]:
dataset_name = train_records[0]["dataset"]
ts = datetime.now().strftime("%Y%m%d-%H%M%S")
run_name = f"{dataset_name}_{cfg.label_key}_{cfg.task_type}_{ts}"

run_dir   = udp.from_project_relative(cfg.runs_dir)   / run_name
model_dir = udp.from_project_relative(cfg.models_dir) / run_name
run_dir.mkdir(parents=True, exist_ok=True)
model_dir.mkdir(parents=True, exist_ok=True)
print(f"run_dir   = {run_dir.relative_to(PROJECT_ROOT)}    (per-epoch logs)")
print(f"model_dir = {model_dir.relative_to(PROJECT_ROOT)}  (best_model goes here)")

# Save the resolved Config alongside the run for reproducibility
from dataclasses import asdict as _asdict
(run_dir / "config.json").write_text(json.dumps(_asdict(cfg), indent=2, default=str))

run_dir   = runs/test/ROG-Art_filled_pause_classification_20260528-175030    (per-epoch logs)
model_dir = models/test/ROG-Art_filled_pause_classification_20260528-175030  (best_model goes here)


927

---

## 16. Load the feature extractor

Single load, reused across both phases.

In [18]:
print(f"loading feature extractor: {cfg.model_name}")
feature_extractor = AutoFeatureExtractor.from_pretrained(cfg.model_name)
print(f"   sampling_rate = {feature_extractor.sampling_rate}")

'[Errno -2] Name or service not known' thrown while requesting HEAD https://huggingface.co/hf-internal-testing/tiny-random-wav2vec2/resolve/main/processor_config.json
Retrying in 1s [Retry 1/5].


loading feature extractor: hf-internal-testing/tiny-random-wav2vec2


OSError: Can't load feature extractor for 'hf-internal-testing/tiny-random-wav2vec2'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'hf-internal-testing/tiny-random-wav2vec2' is the correct path to a directory containing a preprocessor_config.json file

---

## 17. Phase 1 — TRAIN → DEV (development)

Train on TRAIN, evaluate on DEV every epoch. No model saved.

In [ ]:
phase1_results, phase1_best = run_phase(
    phase_name="phase1_dev",
    train_records=train_records, eval_records=dev_records,
    eval_split_name="DEV", save_best_model=False,
    cfg=cfg, run_dir=run_dir, model_dir=model_dir,
    feature_extractor=feature_extractor,
    label2id=label2id, id2label=id2label,
)

---

## 18. Phase 2 — TRAIN + DEV → TEST (final)

Re-train on TRAIN ∪ DEV, evaluate on TEST every epoch. Best epoch's model is saved to `models/<run_name>/best_model/`.

In [ ]:
phase2_results, phase2_best = run_phase(
    phase_name="phase2_test",
    train_records=train_records + dev_records, eval_records=test_records,
    eval_split_name="TEST", save_best_model=True,
    cfg=cfg, run_dir=run_dir, model_dir=model_dir,
    feature_extractor=feature_extractor,
    label2id=label2id, id2label=id2label,
)

---

## 19. Run summary

Final report. Phase-2 numbers are the headline; phase-1 is informational (how things looked on DEV).

In [ ]:
udp.banner(f"RUN SUMMARY: {run_name}")
print(f"task        : {cfg.task_type}")
print(f"label_key   : {cfg.label_key}")
print(f"model       : {cfg.model_name}")
print(f"epochs      : {cfg.num_epochs}")
print(f"run_dir     : {run_dir.relative_to(PROJECT_ROOT)}    (logs)")
print(f"model_dir   : {model_dir.relative_to(PROJECT_ROOT)}  (best model)\n")

print(f"Phase 1 best (DEV)  — epoch {phase1_best['epoch']}:")
for k, v in phase1_best.items():
    if isinstance(v, float):
        print(f"   {k}: {v:.4f}")

print(f"\nPhase 2 best (TEST) — epoch {phase2_best['epoch']}:")
for k, v in phase2_best.items():
    if isinstance(v, float):
        print(f"   {k}: {v:.4f}")

print(f"\nbest model: {(model_dir / 'best_model').relative_to(PROJECT_ROOT)}")

---

## 20. What's next

This run wrote per-epoch logs + a saved best model under `runs/` and `models/`. Chapter 5 (`5_analysis/`) loads run directories like this one and produces:
- boundary / IoU / event-level metrics from `predictions.json`,
- error-analysis CSVs of worst-N records,
- cross-run comparisons.

For a second target on the same dataset, change two lines of Config (`label_key`, maybe `label_order`) and re-run. For a non-binary frame task, also flip `best_metric` away from `frame_f1_positive` (which only makes sense when the last class is the "positive" one).